# CALE Experiment Visualization

This notebook visualizes the JSON report produced by `experiment.py`.

Expected input structure:

```json
{
  "metrics": {...},
  "predictions": [...],
  "stress_summary": {...},
  "stress_tests": [...]
}
```


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("default")


## 1. Load Results

Set `RESULTS_PATH` to the JSON file generated by `experiment.py`.

Example command to create it:

```bash
python3 experiment.py --dataset train.csv --stress --pretty > results.json
```


In [ ]:
RESULTS_PATH = Path("results.json")
OUTDIR = Path("figures")
OUTDIR.mkdir(parents=True, exist_ok=True)

with RESULTS_PATH.open("r", encoding="utf-8") as f:
    report = json.load(f)

report.keys()


## 2. Metrics Table

This table compares evaluator variants, such as direct LLM judge vs full CALE.


In [ ]:
metrics_df = (
    pd.DataFrame(report["metrics"])
    .T
    .reset_index()
    .rename(columns={"index": "variant"})
)
metrics_df


## 3. Bar Plots for Main Metrics

Useful for convergent validity / expert agreement comparisons.


In [ ]:
def plot_metric(metric_name, ylim=(0, 1)):
    if metric_name not in metrics_df.columns:
        print(f"Missing metric: {metric_name}")
        return
    df = metrics_df[["variant", metric_name]].dropna()
    if df.empty:
        print(f"No values for metric: {metric_name}")
        return

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["variant"], df[metric_name])
    ax.set_title(metric_name)
    ax.set_ylabel(metric_name)
    ax.set_ylim(*ylim)
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    fig.tight_layout()
    fig.savefig(OUTDIR / f"metric_{metric_name}.png", dpi=200)
    plt.show()

for metric in ["accuracy", "macro_f1", "checklist_f1", "mean_uncertainty"]:
    plot_metric(metric, ylim=(0, 1))


## 4. Stress Summary

These plots compare robustness under perturbations. Lower invariance score shift and lower label-change rate are better for construct-irrelevant perturbations.


In [ ]:
if "stress_summary" in report:
    stress_summary_df = (
        pd.DataFrame(report["stress_summary"])
        .T
        .reset_index()
        .rename(columns={"index": "variant"})
    )
else:
    stress_summary_df = pd.DataFrame()

stress_summary_df


In [ ]:
def plot_stress_metric(metric_name):
    if stress_summary_df.empty or metric_name not in stress_summary_df.columns:
        print(f"Missing stress metric: {metric_name}")
        return
    df = stress_summary_df[["variant", metric_name]].dropna()
    if df.empty:
        print(f"No values for stress metric: {metric_name}")
        return

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["variant"], df[metric_name])
    ax.set_title(metric_name)
    ax.set_ylabel(metric_name)
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    fig.tight_layout()
    fig.savefig(OUTDIR / f"stress_{metric_name}.png", dpi=200)
    plt.show()

for metric in [
    "invariance_label_change_rate",
    "mean_abs_invariance_score_shift",
    "mean_sensitivity_score_drop",
]:
    plot_stress_metric(metric)


## 5. Score Shift by Perturbation

This shows which perturbations change evaluator scores. For construct-irrelevant perturbations, score shifts should be close to zero.


In [ ]:
stress_df = pd.DataFrame(report.get("stress_tests", []))
stress_df.head()


In [ ]:
if not stress_df.empty:
    for variant in stress_df["variant"].unique():
        sub = stress_df[stress_df["variant"] == variant]
        means = sub.groupby("perturbation")["score_shift"].mean().sort_values()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(means.index, means.values)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"Mean score shift by perturbation: {variant}")
        ax.set_ylabel("score_shift")
        ax.tick_params(axis="x", rotation=30)
        for label in ax.get_xticklabels():
            label.set_ha("right")
        fig.tight_layout()
        fig.savefig(OUTDIR / f"score_shift_{variant}.png", dpi=200)
        plt.show()
else:
    print("No stress_tests found. Run experiment.py with --stress.")


## 6. CALE Subscore Heatmap

This visualizes CALE's construct dimensions for each sample. It is useful for showing interpretability and failure modes.


In [ ]:
prediction_rows = []
for pred in report.get("predictions", []):
    if pred.get("variant") != "full_cale":
        continue
    for dimension, score in pred.get("subscores", {}).items():
        prediction_rows.append({
            "id": pred["id"],
            "dimension": dimension,
            "score": score,
        })

subscore_df = pd.DataFrame(prediction_rows)
subscore_df.head()


In [ ]:
if not subscore_df.empty:
    heatmap_table = subscore_df.pivot(index="id", columns="dimension", values="score")

    fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(heatmap_table))))
    image = ax.imshow(heatmap_table.values, aspect="auto", vmin=0, vmax=1)
    fig.colorbar(image, ax=ax, label="subscore")
    ax.set_xticks(range(len(heatmap_table.columns)))
    ax.set_xticklabels(heatmap_table.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(heatmap_table.index)))
    ax.set_yticklabels(heatmap_table.index)
    ax.set_title("CALE dimension subscores")
    fig.tight_layout()
    fig.savefig(OUTDIR / "cale_subscore_heatmap.png", dpi=200)
    plt.show()
else:
    print("No full_cale subscores found.")


## 7. Export Tables

Save metric and stress tables as CSV for paper tables.


In [ ]:
metrics_df.to_csv(OUTDIR / "metrics_table.csv", index=False)
if not stress_summary_df.empty:
    stress_summary_df.to_csv(OUTDIR / "stress_summary_table.csv", index=False)
if not subscore_df.empty:
    subscore_df.to_csv(OUTDIR / "cale_subscores_long.csv", index=False)

print(f"Saved figures and tables to: {OUTDIR.resolve()}")
